# Financial Market Analytics & Decision Intelligence

## Stock Market Data Analysis and Prediction

This notebook performs exploratory data analysis, financial feature engineering,
baseline comparison, machine-learning modelling, and prediction evaluation.

> **Important:** This project is for analytical and educational purposes.
> Market predictions are inherently uncertain and should not be treated as
> financial advice.


## 1. Objectives

The analysis focuses on:

- Understanding historical OHLCV market behaviour
- Performing data-quality validation
- Engineering financial time-series features
- Establishing a simple baseline
- Training a neural-network regression model
- Evaluating prediction performance
- Comparing actual and predicted prices
- Preparing the model for integration with the Flask dashboard


## 2. Import Libraries

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input

print("TensorFlow version:", tf.__version__)


## 3. Load Dataset

In [ ]:
DATA_PATH = "../data/stock_data.csv"

df = pd.read_csv(DATA_PATH)

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values("Date").reset_index(drop=True)

print("Dataset shape:", df.shape)
df.head()


## 4. Dataset Overview

In [ ]:
print("Rows:", len(df))
print("Columns:", list(df.columns))

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isna().sum())


## 5. Descriptive Statistics

In [ ]:
df.describe().T


## 6. Closing Price History

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(df["Date"], df["Close"])

plt.title("Historical Closing Price")
plt.xlabel("Date")
plt.ylabel("Close Price")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 7. Trading Volume

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(df["Date"], df["Volume"])

plt.title("Historical Trading Volume")
plt.xlabel("Date")
plt.ylabel("Volume")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 8. Return Feature Engineering

In [ ]:
df["Return_1D"] = df["Close"].pct_change()
df["Return_5D"] = df["Close"].pct_change(5)
df["Return_20D"] = df["Close"].pct_change(20)

df[[
    "Date",
    "Close",
    "Return_1D",
    "Return_5D",
    "Return_20D"
]].tail()


## 9. Moving Average Features

In [ ]:
df["MA20"] = df["Close"].rolling(20).mean()
df["MA50"] = df["Close"].rolling(50).mean()
df["MA200"] = df["Close"].rolling(200).mean()

plt.figure(figsize=(14, 7))

plt.plot(df["Date"], df["Close"], label="Close")
plt.plot(df["Date"], df["MA20"], label="MA20")
plt.plot(df["Date"], df["MA50"], label="MA50")
plt.plot(df["Date"], df["MA200"], label="MA200")

plt.title("Price and Moving Averages")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 10. Volatility Analysis

In [ ]:
df["Rolling_Volatility_20D"] = (
    df["Return_1D"]
    .rolling(20)
    .std()
)

df["Annualized_Volatility"] = (
    df["Rolling_Volatility_20D"]
    * np.sqrt(252)
)

df[[
    "Date",
    "Rolling_Volatility_20D",
    "Annualized_Volatility"
]].tail()


## 11. Drawdown Analysis

In [ ]:
running_max = df["Close"].cummax()

df["Drawdown"] = (
    (df["Close"] - running_max)
    / running_max
)

maximum_drawdown = df["Drawdown"].min()

print(
    f"Maximum historical drawdown: "
    f"{maximum_drawdown * 100:.2f}%"
)


## 12. Volume Activity

In [ ]:
df["Average_Volume_20D"] = (
    df["Volume"]
    .rolling(20)
    .mean()
)

df["Volume_Ratio"] = (
    df["Volume"]
    / df["Average_Volume_20D"]
)

df[[
    "Date",
    "Volume",
    "Average_Volume_20D",
    "Volume_Ratio"
]].tail()


## 13. Prediction Target

The target is the **next trading day's closing price**.

The target is shifted by one observation so that today's market information
is used to estimate tomorrow's close.


In [ ]:
df["Target"] = df["Close"].shift(-1)

model_df = df.dropna().copy()

print("Model dataset shape:", model_df.shape)


## 14. Model Features

In [ ]:
FEATURES = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Return_1D",
    "Return_5D",
    "Return_20D",
    "MA20",
    "MA50",
    "MA200",
    "Rolling_Volatility_20D",
    "Volume_Ratio",
]

model_df = model_df.dropna(
    subset=FEATURES + ["Target"]
).copy()

X = model_df[FEATURES]
y = model_df["Target"]

print("Features:", FEATURES)
print("X shape:", X.shape)
print("y shape:", y.shape)


## 15. Chronological Train/Test Split

Financial time-series data must preserve temporal order.

A random train/test split can introduce future information into the training
process, so the data is split chronologically.


In [ ]:
split_index = int(len(model_df) * 0.8)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


## 16. Feature Scaling

In [ ]:
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print("Training data scaled.")
print("Test data transformed using the training scaler.")


## 17. Naive Baseline

In [ ]:
baseline_predictions = X_test["Close"].values

baseline_mae = mean_absolute_error(
    y_test,
    baseline_predictions
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_predictions
    )
)

baseline_r2 = r2_score(
    y_test,
    baseline_predictions
)

print(f"Baseline MAE:  {baseline_mae:.4f}")
print(f"Baseline RMSE: {baseline_rmse:.4f}")
print(f"Baseline R²:   {baseline_r2:.4f}")


## 18. Neural Network Regression Model

In [ ]:
tf.random.set_seed(42)
np.random.seed(42)

model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(1)
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()


## 19. Model Training

In [ ]:
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    shuffle=False,
    verbose=1
)


## 20. Training History

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.tight_layout()
plt.show()


## 21. Generate Predictions

In [ ]:
predictions = model.predict(
    X_test_scaled,
    verbose=0
).flatten()

print("Predictions generated:", len(predictions))


## 22. Model Evaluation

In [ ]:
mae = mean_absolute_error(
    y_test,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predictions
    )
)

r2 = r2_score(
    y_test,
    predictions
)

print(f"Neural Network MAE:  {mae:.4f}")
print(f"Neural Network RMSE: {rmse:.4f}")
print(f"Neural Network R²:   {r2:.4f}")


## 23. Baseline vs Neural Network

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Naive Baseline",
        "Neural Network"
    ],
    "MAE": [
        baseline_mae,
        mae
    ],
    "RMSE": [
        baseline_rmse,
        rmse
    ],
    "R2": [
        baseline_r2,
        r2
    ]
})

comparison


## 24. Actual vs Predicted Prices

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    y_test.values,
    label="Actual"
)

plt.plot(
    predictions,
    label="Predicted"
)

plt.title("Actual vs Predicted Closing Price")
plt.xlabel("Test Observation")
plt.ylabel("Closing Price")
plt.legend()
plt.tight_layout()
plt.show()


## 25. Prediction Error Analysis

In [ ]:
errors = y_test.values - predictions

plt.figure(figsize=(14, 5))

plt.plot(errors)

plt.axhline(
    0,
    linestyle="--"
)

plt.title("Prediction Errors")
plt.xlabel("Test Observation")
plt.ylabel("Actual - Predicted")
plt.tight_layout()
plt.show()


## 26. Error Statistics

In [ ]:
error_summary = pd.Series(errors).describe()

error_summary


## 27. Latest Next-Day Prediction

In [ ]:
latest_features = X.iloc[[-1]]

latest_scaled = scaler.transform(
    latest_features
)

next_day_prediction = float(
    model.predict(
        latest_scaled,
        verbose=0
    )[0][0]
)

current_price = float(
    df["Close"].iloc[-1]
)

predicted_change = (
    (next_day_prediction / current_price) - 1
) * 100

print(f"Current close: {current_price:.2f}")
print(
    f"Predicted next close: "
    f"{next_day_prediction:.2f}"
)
print(
    f"Predicted change: "
    f"{predicted_change:.2f}%"
)



# 28. Model Diagnostics

The neural network is compared against the naive persistence
baseline to determine whether the learned model provides genuine
out-of-sample predictive value.

A model should only be integrated into the dashboard if it
demonstrates meaningful performance relative to the baseline.


In [ ]:

diagnostic_results = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Naive Baseline": [
        baseline_mae,
        baseline_rmse,
        baseline_r2,
    ],
    "Neural Network": [
        mae,
        rmse,
        r2,
    ],
})

diagnostic_results



## 28.1 Prediction Statistics

The prediction distribution is inspected to identify systematic
bias, excessive compression, or unstable predictions.


In [ ]:

prediction_summary = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": predictions,
})

print("Actual price statistics:")
display(prediction_summary["Actual"].describe())

print("\nPredicted price statistics:")
display(prediction_summary["Predicted"].describe())

print("\nPrediction bias:")
print(
    f"Mean prediction error: "
    f"{errors.mean():.4f}"
)

print(
    f"Mean absolute error: "
    f"{np.abs(errors).mean():.4f}"
)



## 28.2 Prediction vs Actual Scatter

A well-calibrated regression model should produce predictions
that approximately follow the diagonal relationship between
actual and predicted values.


In [ ]:

plt.figure(figsize=(8, 8))

plt.scatter(
    y_test,
    predictions,
    alpha=0.7
)

minimum = min(
    y_test.min(),
    predictions.min()
)

maximum = max(
    y_test.max(),
    predictions.max()
)

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--"
)

plt.xlabel("Actual Closing Price")
plt.ylabel("Predicted Closing Price")
plt.title("Predicted vs Actual Closing Price")

plt.tight_layout()
plt.show()



## 28.3 Baseline Advantage

The percentage improvement or deterioration of the neural network
relative to the naive baseline is calculated for each evaluation
metric.


In [ ]:

mae_change = (
    (mae - baseline_mae)
    / baseline_mae
    * 100
)

rmse_change = (
    (rmse - baseline_rmse)
    / baseline_rmse
    * 100
)

print(
    f"MAE change vs baseline: "
    f"{mae_change:+.2f}%"
)

print(
    f"RMSE change vs baseline: "
    f"{rmse_change:+.2f}%"
)

if mae < baseline_mae and rmse < baseline_rmse:
    print(
        "The neural network outperforms the naive baseline."
    )
else:
    print(
        "The neural network does not outperform "
        "the naive baseline."
    )



## 28.4 Model Diagnostic Conclusion

The evaluation indicates whether the neural network should be
retained as the primary forecasting model or whether the modelling
formulation should be improved before production integration.


In [ ]:

if mae < baseline_mae and rmse < baseline_rmse and r2 > baseline_r2:
    diagnostic_conclusion = (
        "The neural network demonstrates improved "
        "out-of-sample performance over the naive baseline."
    )
else:
    diagnostic_conclusion = (
        "The neural network currently underperforms the "
        "naive persistence baseline. The forecasting formulation "
        "should be improved before deployment."
    )

print(diagnostic_conclusion)


## 28. Model Limitations

This model should be interpreted as an analytical experiment rather than a
reliable trading system.

Important limitations include:

- Financial markets are highly non-stationary.
- Historical OHLCV data does not capture all market information.
- News, macroeconomic events, sentiment, and liquidity can affect prices.
- Regression accuracy does not imply profitable trading performance.
- The model predicts price levels rather than actual trading returns.
- Longer-horizon forecasts accumulate uncertainty.


## 29. Conclusion

The analysis establishes a complete machine-learning workflow:

**Data → Validation → Feature Engineering → Chronological Split → Scaling →
Baseline → Neural Network → Evaluation → Error Analysis**

The resulting model can serve as the machine-learning component of the Flask
Financial Market Analytics and Decision Intelligence Dashboard.



# 30. Alternative Forecasting Formulation — Next-Day Return

The previous experiment predicted the absolute next-day closing price.

This section reformulates the problem as **next-day return prediction**.

Instead of directly predicting:

`Tomorrow's Close`

the model predicts:

`Tomorrow's Return`

The predicted return can then be converted back into a price estimate:

`Predicted Price = Current Price × (1 + Predicted Return)`

This formulation is tested independently against the existing
persistence baseline.


In [ ]:

# Create next-day return target

return_df = df.copy()

return_df["Target_Return"] = (
    return_df["Close"].shift(-1) / return_df["Close"]
) - 1

return_df = return_df.dropna().copy()

print("Return forecasting dataset:", return_df.shape)
return_df[["Date", "Close", "Target_Return"]].tail()



## 31. Return Model Features

The same engineered market features are retained so that the experiment
changes the forecasting target rather than introducing an unrelated
feature set.


In [ ]:

RETURN_FEATURES = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Return_1D",
    "Return_5D",
    "Return_20D",
    "MA20",
    "MA50",
    "MA200",
    "Rolling_Volatility_20D",
    "Volume_Ratio",
]

return_df = return_df.dropna(
    subset=RETURN_FEATURES + ["Target_Return"]
).copy()

X_return = return_df[RETURN_FEATURES]
y_return = return_df["Target_Return"]

split_return = int(len(return_df) * 0.8)

X_return_train = X_return.iloc[:split_return].copy()
X_return_test = X_return.iloc[split_return:].copy()

y_return_train = y_return.iloc[:split_return].copy()
y_return_test = y_return.iloc[split_return:].copy()

print("Training samples:", len(X_return_train))
print("Testing samples:", len(X_return_test))



## 32. Scale Return Model Features

The scaler is fitted only on the chronological training set to prevent
future information from leaking into the training process.


In [ ]:

return_scaler = MinMaxScaler()

X_return_train_scaled = return_scaler.fit_transform(
    X_return_train
)

X_return_test_scaled = return_scaler.transform(
    X_return_test
)

print("Return-model features scaled successfully.")



## 33. Return Prediction Neural Network


In [ ]:

tf.random.set_seed(42)
np.random.seed(42)

return_model = Sequential([
    Input(shape=(X_return_train_scaled.shape[1],)),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(1)
])

return_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

return_model.summary()



## 34. Train Return Model


In [ ]:

return_history = return_model.fit(
    X_return_train_scaled,
    y_return_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    shuffle=False,
    verbose=1
)



## 35. Generate Return Predictions


In [ ]:

return_predictions = return_model.predict(
    X_return_test_scaled,
    verbose=0
).flatten()

print(
    "Return predictions generated:",
    len(return_predictions)
)



## 36. Return Model Evaluation


In [ ]:

return_mae = mean_absolute_error(
    y_return_test,
    return_predictions
)

return_rmse = np.sqrt(
    mean_squared_error(
        y_return_test,
        return_predictions
    )
)

return_r2 = r2_score(
    y_return_test,
    return_predictions
)

print(f"Return Model MAE:  {return_mae:.6f}")
print(f"Return Model RMSE: {return_rmse:.6f}")
print(f"Return Model R²:   {return_r2:.6f}")



## 37. Directional Accuracy

For financial forecasting, correctly predicting the direction of movement
can be more informative than absolute price error.

A prediction is considered directionally correct when the predicted return
and actual return have the same sign.


In [ ]:

actual_direction = np.sign(
    y_return_test.values
)

predicted_direction = np.sign(
    return_predictions
)

directional_accuracy = np.mean(
    actual_direction == predicted_direction
)

print(
    f"Directional Accuracy: "
    f"{directional_accuracy * 100:.2f}%"
)



## 38. Return Prediction vs Persistence Baseline

For the return formulation, a natural persistence baseline predicts a
zero return for the next trading day.


In [ ]:

zero_return_predictions = np.zeros(
    len(y_return_test)
)

zero_return_mae = mean_absolute_error(
    y_return_test,
    zero_return_predictions
)

zero_return_rmse = np.sqrt(
    mean_squared_error(
        y_return_test,
        zero_return_predictions
    )
)

zero_return_r2 = r2_score(
    y_return_test,
    zero_return_predictions
)

print(f"Zero-return baseline MAE:  {zero_return_mae:.6f}")
print(f"Zero-return baseline RMSE: {zero_return_rmse:.6f}")
print(f"Zero-return baseline R²:   {zero_return_r2:.6f}")



## 39. Return Model Comparison


In [ ]:

return_comparison = pd.DataFrame({
    "Model": [
        "Zero-Return Baseline",
        "Neural Network"
    ],
    "MAE": [
        zero_return_mae,
        return_mae
    ],
    "RMSE": [
        zero_return_rmse,
        return_rmse
    ],
    "R2": [
        zero_return_r2,
        return_r2
    ],
    "Directional_Accuracy": [
        50.0,
        directional_accuracy * 100
    ]
})

return_comparison



## 40. Actual vs Predicted Returns


In [ ]:

plt.figure(figsize=(14, 6))

plt.plot(
    y_return_test.values,
    label="Actual Return"
)

plt.plot(
    return_predictions,
    label="Predicted Return"
)

plt.axhline(
    0,
    linestyle="--"
)

plt.title("Actual vs Predicted Next-Day Returns")
plt.xlabel("Test Observation")
plt.ylabel("Return")
plt.legend()
plt.tight_layout()
plt.show()



## 41. Return Model Assessment

The return-based neural network should only be considered for dashboard
integration if it demonstrates meaningful improvement over the baseline.

The decision should be based on:

- MAE
- RMSE
- R²
- Directional Accuracy
- Stability of the predictions

A neural network should not be selected merely because it is more complex.


In [ ]:

print("Return Forecasting Assessment")
print("=" * 50)

print(f"Zero-return baseline MAE: {zero_return_mae:.6f}")
print(f"Return model MAE:         {return_mae:.6f}")
print()

print(f"Zero-return baseline RMSE: {zero_return_rmse:.6f}")
print(f"Return model RMSE:         {return_rmse:.6f}")
print()

print(f"Return model R²:           {return_r2:.6f}")
print(
    f"Directional accuracy:     "
    f"{directional_accuracy * 100:.2f}%"
)

if return_mae < zero_return_mae and directional_accuracy > 0.50:
    print(
        "\nRESULT: Return model shows improvement over "
        "the zero-return baseline."
    )
else:
    print(
        "\nRESULT: Return model does not currently "
        "provide sufficient evidence of improvement."
    )



# 42. Gradient Boosting Benchmark

The previous feed-forward neural-network experiments did not outperform
simple persistence baselines.

This section evaluates a tree-based gradient boosting model on the same
engineered financial features.

The objective is to determine whether a nonlinear tabular model can extract
predictive information that the neural network failed to capture.


In [ ]:

from sklearn.ensemble import HistGradientBoostingRegressor

print("Gradient Boosting benchmark initialized.")



## 43. Chronological Gradient Boosting Dataset

The original next-day closing-price formulation is retained for direct
comparison with the earlier baseline and neural network experiments.


In [ ]:

gb_df = df.copy()

gb_df["Target"] = gb_df["Close"].shift(-1)

gb_df = gb_df.dropna(
    subset=FEATURES + ["Target"]
).copy()

X_gb = gb_df[FEATURES]
y_gb = gb_df["Target"]

gb_split = int(len(gb_df) * 0.8)

X_gb_train = X_gb.iloc[:gb_split].copy()
X_gb_test = X_gb.iloc[gb_split:].copy()

y_gb_train = y_gb.iloc[:gb_split].copy()
y_gb_test = y_gb.iloc[gb_split:].copy()

print("Gradient Boosting training samples:", len(X_gb_train))
print("Gradient Boosting testing samples:", len(X_gb_test))



## 44. Train Gradient Boosting Regressor

Gradient boosting is evaluated as a nonlinear tabular-data model.

Unlike the neural network, tree-based gradient boosting does not require
feature scaling.


In [ ]:

gb_model = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_iter=300,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    random_state=42
)

gb_model.fit(
    X_gb_train,
    y_gb_train
)

print("Gradient Boosting model trained successfully.")



## 45. Generate Gradient Boosting Predictions


In [ ]:

gb_predictions = gb_model.predict(
    X_gb_test
)

print(
    "Predictions generated:",
    len(gb_predictions)
)



## 46. Gradient Boosting Evaluation


In [ ]:

gb_mae = mean_absolute_error(
    y_gb_test,
    gb_predictions
)

gb_rmse = np.sqrt(
    mean_squared_error(
        y_gb_test,
        gb_predictions
    )
)

gb_r2 = r2_score(
    y_gb_test,
    gb_predictions
)

print(f"Gradient Boosting MAE:  {gb_mae:.4f}")
print(f"Gradient Boosting RMSE: {gb_rmse:.4f}")
print(f"Gradient Boosting R²:   {gb_r2:.4f}")



## 47. Gradient Boosting Directional Accuracy

Directional accuracy measures whether the model correctly identifies the
direction of the next-day price movement.


In [ ]:

actual_price_change = (
    y_gb_test.values
    - X_gb_test["Close"].values
)

predicted_price_change = (
    gb_predictions
    - X_gb_test["Close"].values
)

gb_directional_accuracy = np.mean(
    np.sign(actual_price_change)
    == np.sign(predicted_price_change)
)

print(
    f"Gradient Boosting Directional Accuracy: "
    f"{gb_directional_accuracy * 100:.2f}%"
)



## 48. Baseline vs Neural Network vs Gradient Boosting


In [ ]:

model_comparison_extended = pd.DataFrame({
    "Model": [
        "Naive Persistence",
        "Dense Neural Network",
        "Gradient Boosting"
    ],
    "MAE": [
        baseline_mae,
        mae,
        gb_mae
    ],
    "RMSE": [
        baseline_rmse,
        rmse,
        gb_rmse
    ],
    "R2": [
        baseline_r2,
        r2,
        gb_r2
    ],
    "Directional_Accuracy": [
        np.mean(
            np.sign(
                y_test.values
                - X_test["Close"].values
            )
            == np.sign(
                X_test["Close"].values
                - X_test["Close"].values
            )
        ) * 100,
        np.nan,
        gb_directional_accuracy * 100
    ]
})

model_comparison_extended



## 49. Actual vs Gradient Boosting Predictions


In [ ]:

plt.figure(figsize=(14, 6))

plt.plot(
    y_gb_test.values,
    label="Actual"
)

plt.plot(
    gb_predictions,
    label="Gradient Boosting"
)

plt.title(
    "Actual vs Gradient Boosting Predicted Closing Price"
)

plt.xlabel("Test Observation")
plt.ylabel("Closing Price")

plt.legend()
plt.tight_layout()
plt.show()



## 50. Gradient Boosting Model Decision

The model is considered a candidate for deployment only if it provides
meaningful improvement over the persistence baseline.

The primary selection criteria are:

- Lower MAE
- Lower RMSE
- Higher R²
- Useful directional accuracy


In [ ]:

print("Gradient Boosting Assessment")
print("=" * 50)

print(f"Baseline MAE:          {baseline_mae:.4f}")
print(f"Gradient Boosting MAE: {gb_mae:.4f}")
print()

print(f"Baseline RMSE:          {baseline_rmse:.4f}")
print(f"Gradient Boosting RMSE: {gb_rmse:.4f}")
print()

print(f"Baseline R²:          {baseline_r2:.4f}")
print(f"Gradient Boosting R²: {gb_r2:.4f}")
print()

print(
    f"Gradient Boosting Directional Accuracy: "
    f"{gb_directional_accuracy * 100:.2f}%"
)

if gb_mae < baseline_mae and gb_rmse < baseline_rmse:
    print(
        "\nRESULT: Gradient Boosting improves over "
        "the persistence baseline."
    )
else:
    print(
        "\nRESULT: Gradient Boosting does not currently "
        "outperform the persistence baseline."
    )


In [ ]:
# Correlation analysis for numerical market variables

correlation_columns = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Return_1D",
    "Return_5D",
    "Return_20D",
    "MA20",
    "MA50",
    "MA200",
    "Rolling_Volatility_20D",
    "Volume_Ratio",
]

correlation_matrix = (
    df[correlation_columns]
    .corr()
)

display(
    correlation_matrix.round(2)
)


In [ ]:
# Visualize feature correlations

plt.figure(figsize=(12, 9))

plt.imshow(
    correlation_matrix,
    aspect="auto"
)

plt.colorbar(
    label="Correlation"
)

plt.xticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns,
    rotation=90
)

plt.yticks(
    range(len(correlation_matrix.index)),
    correlation_matrix.index
)

plt.title(
    "Market Feature Correlation Matrix"
)

plt.tight_layout()
plt.show()
